# Module 1: LLM 推理基础

## 学习目标
- 理解 LLM 推理的基本流程
- 了解 Prefill 和 Decode 两个阶段
- 掌握 KV Cache 的基本概念
- 理解为什么需要 mini-sglang 这样的推理框架

---

## 1.1 LLM 推理的两个阶段

LLM (Large Language Model) 推理分为两个关键阶段：

### Prefill 阶段 (预填充)
- **输入**: 用户的完整 prompt (例如: "What is AI?")
- **处理**: 一次性处理所有输入 token
- **特点**: 计算密集型 (compute-bound), 可以并行处理
- **输出**: 所有 token 的 KV Cache + 第一个输出 token

### Decode 阶段 (解码)
- **输入**: 上一步生成的 token (单个 token)
- **处理**: 逐个生成 token，直到遇到 EOS 或达到最大长度
- **特点**: 内存密集型 (memory-bound), 每次只处理一个 token
- **输出**: 下一个 token

```
┌─────────────────────────────────────────────────────────────┐
│  Prefill 阶段                                               │
│  "What is AI?" → [Token1, Token2, Token3, Token4]          │
│                                                             │
│  ┌─────┐ ┌─────┐ ┌─────┐ ┌─────┐                          │
│  │ T1  │ │ T2  │ │ T3  │ │ T4  │  → 并行处理              │
│  └─────┘ └─────┘ └─────┘ └─────┘                          │
│       ↓       ↓       ↓       ↓                            │
│  [KV Cache for all tokens] + [First output token]         │
└─────────────────────────────────────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│  Decode 阶段                                                │
│                                                             │
│  Step 1: [Output1] → 使用 KV Cache → [Output2]             │
│  Step 2: [Output2] → 更新 KV Cache → [Output3]             │
│  Step 3: [Output3] → 更新 KV Cache → [Output4]             │
│  ...                                                        │
│  Step N: [OutputN] → 更新 KV Cache → [EOS] (结束)          │
└─────────────────────────────────────────────────────────────┘
```

## 1.2 什么是 KV Cache?

在 Transformer 的 Self-Attention 中，对于每个 token，我们需要计算:
- Q (Query): 当前 token 的查询向量
- K (Key): 用于匹配的键向量
- V (Value): 包含信息的值向量

**问题**: 在 Decode 阶段，每生成一个新 token，都需要与之前所有 token 进行 attention 计算。

**解决方案**: KV Cache - 缓存之前所有 token 的 K 和 V，避免重复计算。

```python
# 没有 KV Cache (每次都要重新计算)
for step in range(max_tokens):
    all_tokens = input_tokens + generated_tokens
    Q, K, V = compute_qkv(all_tokens)  # O(n) 计算
    output = attention(Q, K, V)         # O(n²) 计算
    
# 使用 KV Cache (只计算新 token)
K_cache, V_cache = [], []
for step in range(max_tokens):
    new_token = generated_tokens[-1]
    q, k, v = compute_qkv(new_token)    # O(1) 计算
    K_cache.append(k)
    V_cache.append(v)
    output = attention(q, K_cache, V_cache)  # O(n) 计算
```

## 1.3 KV Cache 的内存占用

KV Cache 的大小计算公式:

```
KV Cache Size = 2 (K+V) × num_layers × num_kv_heads × head_dim × seq_len × batch_size × dtype_size
```

以 Llama-7B 为例:
- num_layers = 32
- num_kv_heads = 32
- head_dim = 128
- dtype_size = 2 (bfloat16)

对于一个 4096 长度的序列:
```
2 × 32 × 32 × 128 × 4096 × 2 = 2.1 GB
```

这就是为什么 KV Cache 管理如此重要！

In [ ]:
# 计算 KV Cache 大小的函数
def compute_kv_cache_size(
    num_layers: int,
    num_kv_heads: int,
    head_dim: int,
    seq_len: int,
    batch_size: int = 1,
    dtype_size: int = 2  # bfloat16
) -> float:
    """计算 KV Cache 大小 (GB)"""
    size_bytes = 2 * num_layers * num_kv_heads * head_dim * seq_len * batch_size * dtype_size
    return size_bytes / (1024 ** 3)  # Convert to GB

# Qwen3-0.6B 示例
print("Qwen3-0.6B (seq_len=4096):")
print(f"  KV Cache Size: {compute_kv_cache_size(28, 4, 128, 4096):.2f} GB")

# Llama-3-8B 示例
print("\nLlama-3-8B (seq_len=4096):")
print(f"  KV Cache Size: {compute_kv_cache_size(32, 8, 128, 4096):.2f} GB")

# Qwen3-32B 示例
print("\nQwen3-32B (seq_len=4096):")
print(f"  KV Cache Size: {compute_kv_cache_size(64, 8, 128, 4096):.2f} GB")

## 1.4 为什么需要 Mini-SGLang 这样的推理框架?

直接使用 HuggingFace Transformers 进行推理有以下问题:

### 问题 1: 低效的 Batching
- 朴素实现需要等待所有请求完成才能处理下一批
- 短请求需要等待长请求，造成资源浪费

### 问题 2: KV Cache 内存管理
- 需要预分配最大长度的 KV Cache
- 无法动态回收已完成请求的内存

### 问题 3: 重复计算
- 相同前缀的请求需要重复计算 KV Cache
- 例如: 多个用户问相同的系统 prompt

### Mini-SGLang 的解决方案:

| 问题 | 解决方案 |
|------|----------|
| 低效 Batching | Continuous Batching (动态批处理) |
| KV Cache 管理 | Paged Attention + 动态分配 |
| 重复计算 | Radix Cache (前缀复用) |
| CPU 开销 | CUDA Graph + Overlap Scheduling |
| 大模型 | Tensor Parallelism (多 GPU) |

## 1.5 Mini-SGLang 系统架构概览

```
┌─────────────────────────────────────────────────────────────┐
│  用户请求 (HTTP API)                                        │
└──────────────────────┬──────────────────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────────────────┐
│  API Server (FastAPI)                                       │
│  - OpenAI 兼容的 /v1/chat/completions 接口                  │
└──────────────────────┬──────────────────────────────────────┘
                       │ ZMQ
                       ▼
┌─────────────────────────────────────────────────────────────┐
│  Tokenizer Worker                                           │
│  - 文本 → Token IDs (Tokenization)                          │
│  - Token IDs → 文本 (Detokenization)                        │
└──────────────────────┬──────────────────────────────────────┘
                       │ ZMQ
                       ▼
┌─────────────────────────────────────────────────────────────┐
│  Scheduler (调度器)                                         │
│  - 管理请求队列                                              │
│  - 决定 Prefill/Decode 批次                                 │
│  - 管理 KV Cache 分配                                       │
└──────────────────────┬──────────────────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────────────────┐
│  Engine (推理引擎)                                          │
│  - 模型前向传播                                              │
│  - Token 采样                                               │
│  - CUDA Graph 管理                                          │
└─────────────────────────────────────────────────────────────┘
```

## 1.6 动手实验: 观察 Prefill vs Decode

让我们用简单的代码来模拟 Prefill 和 Decode 的区别。

In [ ]:
import torch
import torch.nn as nn
import time

# 模拟一个简单的 Attention 层
class SimpleAttention(nn.Module):
    def __init__(self, hidden_size: int, num_heads: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v_proj = nn.Linear(hidden_size, hidden_size)
        self.o_proj = nn.Linear(hidden_size, hidden_size)
        
        # KV Cache
        self.k_cache = None
        self.v_cache = None
    
    def reset_cache(self):
        self.k_cache = None
        self.v_cache = None
    
    def forward(self, x: torch.Tensor, use_cache: bool = True):
        batch_size, seq_len, _ = x.shape
        
        # 计算 Q, K, V
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        
        # 使用 KV Cache
        if use_cache:
            if self.k_cache is not None:
                k = torch.cat([self.k_cache, k], dim=1)
                v = torch.cat([self.v_cache, v], dim=1)
            self.k_cache = k
            self.v_cache = v
        
        # Reshape for multi-head attention
        q = q.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Attention
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_weights = torch.softmax(attn_weights, dim=-1)
        attn_output = torch.matmul(attn_weights, v)
        
        # Reshape and output projection
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.hidden_size)
        return self.o_proj(attn_output)

# 创建模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
hidden_size = 256
num_heads = 8
attention = SimpleAttention(hidden_size, num_heads).to(device)

print(f"Device: {device}")

In [ ]:
# 模拟 Prefill 阶段
input_len = 512
x_prefill = torch.randn(1, input_len, hidden_size).to(device)

attention.reset_cache()

# 预热
for _ in range(3):
    with torch.no_grad():
        _ = attention(x_prefill)
    attention.reset_cache()

# 计时
if device.type == "cuda":
    torch.cuda.synchronize()
start = time.time()

with torch.no_grad():
    output_prefill = attention(x_prefill)

if device.type == "cuda":
    torch.cuda.synchronize()
prefill_time = time.time() - start

print(f"Prefill 阶段 ({input_len} tokens):")
print(f"  输入形状: {x_prefill.shape}")
print(f"  输出形状: {output_prefill.shape}")
print(f"  KV Cache 形状: K={attention.k_cache.shape}, V={attention.v_cache.shape}")
print(f"  耗时: {prefill_time*1000:.2f} ms")

In [ ]:
# 模拟 Decode 阶段 (不重置 cache，继续使用)
num_decode_steps = 50
decode_times = []

for step in range(num_decode_steps):
    # 每次只输入一个新 token
    x_decode = torch.randn(1, 1, hidden_size).to(device)
    
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    
    with torch.no_grad():
        output_decode = attention(x_decode)
    
    if device.type == "cuda":
        torch.cuda.synchronize()
    decode_times.append(time.time() - start)

avg_decode_time = sum(decode_times) / len(decode_times)
print(f"\nDecode 阶段 ({num_decode_steps} steps):")
print(f"  每步输入形状: (1, 1, {hidden_size})")
print(f"  每步输出形状: {output_decode.shape}")
print(f"  最终 KV Cache 形状: K={attention.k_cache.shape}")
print(f"  平均每步耗时: {avg_decode_time*1000:.3f} ms")
print(f"  总共生成了 {input_len + num_decode_steps} tokens 的 KV Cache")

In [ ]:
# 对比：没有 KV Cache 的情况
attention.reset_cache()

no_cache_times = []
for step in range(num_decode_steps):
    # 每次都需要输入所有之前的 tokens
    all_tokens = torch.randn(1, input_len + step + 1, hidden_size).to(device)
    
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    
    with torch.no_grad():
        _ = attention(all_tokens, use_cache=False)
    
    if device.type == "cuda":
        torch.cuda.synchronize()
    no_cache_times.append(time.time() - start)
    attention.reset_cache()

avg_no_cache_time = sum(no_cache_times) / len(no_cache_times)
print(f"\n没有 KV Cache 的情况:")
print(f"  平均每步耗时: {avg_no_cache_time*1000:.3f} ms")
print(f"  加速比: {avg_no_cache_time / avg_decode_time:.1f}x")

## 1.7 小结

在这个模块中，我们学习了:

1. **LLM 推理的两个阶段**:
   - Prefill: 处理完整输入，计算密集
   - Decode: 逐 token 生成，内存密集

2. **KV Cache**:
   - 缓存之前 token 的 K 和 V 向量
   - 避免重复计算，显著提升 Decode 速度
   - 内存占用与序列长度成正比

3. **推理框架的必要性**:
   - 动态批处理提高 GPU 利用率
   - 智能 KV Cache 管理节省内存
   - 前缀复用减少重复计算

---

**下一步**: [Module 2: 核心数据结构](./02_core_data_structures.ipynb) - 学习 Mini-SGLang 的 `Req`, `Batch`, `Context` 等核心类。